# 00 Preliminary analysis

**Purpose:** get familiar with the supplied BNPL data and identify decisions for the ETL layers (Week 6).

| | Details |
|---|---|
| Inputs | Original BNPL sources in `tables/landing/project-2-bnpl-tables-part*`; ABS extracts in `tables/landing/abs_2021` |
| Outputs | Inline inventory, schemas, value profiles, join checks and proposed layer design; no data files written |
| Run | Run all cells in order using Python with `pandas`, `numpy` and `pyarrow` (available in `mast30034`). Start from the repository root or `code/notebooks/`. |
| Scope | All source rows are profiled. Transaction partitions are read one at a time. Examples are previews, not statistical samples. |

The [project spec](../../MAST30034_Project_2_Buy_Now_Pay_Later_Spec.zip) asks for preliminary exploration, an ETL start and an external-data candidate in Week 6.

**Note:** merchant ABNs are stored as an index; consumer data use a pipe delimiter; transaction dates partitioned into snapshots by timeframe (given in folder name); the last snapshot extends beyond folder name timeframe; customer-order fraud data duplicates exist; some order ABNs have no merchant record. These affect ingestion and joins before any ranking work.

## 1. Find and load the sources

Small reference tables fit in pandas. Keep postcodes as text so they are treated as geographic codes. Reset the merchant index **in memory** to join on ABN.

In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display, Markdown

# Set display options
pd.set_option("display.max_columns", 12)
pd.set_option("display.max_colwidth", 110)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [2]:
# Set directory paths
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "tables/landing").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Run from the repository root or its code/notebooks/ folder.")
LANDING = ROOT / "tables/landing"
PART1 = LANDING / "project-2-bnpl-tables-part1"

# Read merchant and consumer data from Part 1
merchants = pd.read_parquet(PART1 / "tbl_merchants.parquet").reset_index() # reset index in memory
consumers = pd.read_csv(PART1 / "tbl_consumer.csv", sep="|", dtype={"postcode": "string"}) # pipe-delim instead of comma-delim
users = pd.read_parquet(PART1 / "consumer_user_details.parquet")
consumer_fraud = pd.read_csv(PART1 / "consumer_fraud_probability.csv")
merchant_fraud = pd.read_csv(PART1 / "merchant_fraud_probability.csv")
small_tables = {"merchants": merchants, "consumers": consumers, "users": users,
                "consumer_fraud": consumer_fraud, "merchant_fraud": merchant_fraud}

# Read transaction snapshots using wildcard
snapshots = sorted(LANDING.glob("*/transactions*_snapshot"))
if not snapshots:
    raise FileNotFoundError("No transaction snapshots found in landing.")

# Short display labels derived from full source names
snapshot_labels = {p.name: p.parent.name.replace("project-2-bnpl-tables-", "") for p in snapshots}

Inspect each dataset:  merchants, consumers, users, consumer_fraud, merchant_fraud, and transactions

In [3]:
# Find transaction row counts per order-datetime
file_rows = []
for snapshot in snapshots:
    for path in sorted(snapshot.rglob("*.parquet")):
        date = next(p.split("=", 1)[1] for p in path.parts if p.startswith("order_datetime="))
        file_rows.append({"snapshot": snapshot.name, "path": path, "date": date,
                          "rows": pq.ParquetFile(path).metadata.num_rows})
files = pd.DataFrame(file_rows)

grains = {"merchants": "merchant (ABN)", "consumers": "consumer (consumer_id)",
          "users": "user_id ↔ consumer_id mapping",
          "consumer_fraud": "user-date fraud probability",
          "merchant_fraud": "merchant-date fraud probability"}
inventory = [{"source": name, "rows": len(df), "columns": df.shape[1],
              "one row represents": grains[name]} for name, df in small_tables.items()]
for name, group in files.groupby("snapshot", sort=True):
    inventory.append({"source": name, "rows": group.rows.sum(), 
                      "columns": len(set(pq.read_schema(group.iloc[0]["path"]).names) | {"order_datetime"}), # read no. columns
                      "one row represents": "order/transaction"})
display(pd.DataFrame(inventory).set_index("source"))
print(f"Total transaction rows: {files.rows.sum():,}")

,rows,columns,one row represents
source,,,
merchants,4026,3,merchant (ABN)
consumers,499999,6,consumer (consumer_id)
users,499999,2,user_id ↔ consumer_id mapping
consumer_fraud,34864,3,user-date fraud probability
merchant_fraud,114,3,merchant-date fraud probability
transactions_20210228_20210827_snapshot,3643266,5,order/transaction
transactions_20210828_20220227_snapshot,4508106,5,order/transaction
transactions_20220228_20220828_snapshot,6044133,5,order/transaction


Total transaction rows: 14,195,505


### How to read the profiles

Each table below combines its column meaning, loaded pandas type, missing count, distinct count and observed range or example values. All counts use the full table. **Distinct excludes nulls.** Numeric identifier describe numeric range; names and addresses are synthetic free text. A blank-string check supplements pandas null detection.

In [4]:
def profile(df):
    """ 
    Takes a dataframe with a `meanings` dictionary specifying in plain English what each column is
    For each column, display its type, whether there are NULL or blank (strings), no. of distinct entries, and numeric ranges (or categorical example)
    i.e. profile the dataframe by its columns
    """
    rows = []
    for col in df.columns:
        s = df[col]
        values = s.dropna()

        # Set range/values to display depending on data type
        if pd.api.types.is_numeric_dtype(s):
            observed = f"{s.min():,.2f} → {s.max():,.2f}" if s.dtype.kind == "f" else f"{s.min()} → {s.max()}"
        elif col == "order_datetime":
            dates = pd.to_datetime(s, errors="coerce")
            observed = f"{dates.min().date()} → {dates.max().date()}; {int(dates.isna().sum())} invalid/null dates"
        elif s.nunique() <= 12:
            observed = ", ".join(sorted(values.astype(str).unique()))
        else:
            observed = "Examples: " + " | ".join(str(v)[:65] for v in values.unique()[:2])

        # Display rows
        rows.append({"column": col, "dtype": str(s.dtype), "count": len(s),
                     "null": s.isna().sum(), "blank": s.astype("string").str.strip().eq("").sum(),
                     "distinct": s.nunique(), "range / values": observed})

    display(pd.DataFrame(rows).set_index("column"))

## 2. Merchants

**Row:** one merchant. `merchant_abn` is the candidate key. `pd.read_parquet()` initially returns 4,026 rows × 2 columns **plus** the named ABN index; the working table exposes 3 columns. There is no merchant-location column in this source.

`tags` bundles a business category, a letter code and a take rate. Brackets and capitalisation vary, so splitting on every comma would break categories containing commas.

In [5]:
# Show merchants data
display(merchants.head(3))

# Profile merchants data by column
profile(merchants)

,merchant_abn,name,tags
0,10023283211,Felis Limited,"((furniture, home furnishings and equipment shops, and manufacturers, except appliances), (e), (take rate:..."
1,10142254217,Arcu Ac Orci Corporation,"([cable, satellite, and otHer pay television and radio services], [b], [take rate: 4.22])"
2,10165489824,Nunc Sed Company,"([jewelry, watch, clock, and silverware shops], [b], [take rate: 4.40])"


,dtype,count,null,blank,distinct,range / values
column,,,,,,
merchant_abn,int64,4026,0,0,4026,10023283211 → 99990536339
name,object,4026,0,0,4026,Examples: Felis Limited | Arcu Ac Orci Corporation
tags,object,4026,0,0,3954,"Examples: ((furniture, home furnishings and equipment shops, and manufactur | ([cable, satellite, and otHe..."


### Quick look inside `tags`

This is an exploratory extraction into a separate table. Treat the letter code's meaning and take-rate units as unconfirmed: the spec describes a percentage revenue share, but supplies no field-level dictionary. Check these before using them in revenue features. Category labels are lowercased, repeated whitespace is collapsed, commas/colons have no preceding space and one following space, and hyphens have one space on each side. This merges formatting variants without deleting meaningful punctuation or guessing spelling corrections. Original `tags` remain unchanged.

The general format for the `tags` column is: $$((\text{category description}, \cdots), (\text{letter code}), (\text{take rate: } \cdots))$$
But the inner brackets can either be [] OR (), used inconsistently. Examples: 
- ((furniture, home furnishings and equipment shops, and manufacturers, except appliances), (e), (take rate: 0.18))
- ([cable, satellite, and otHer pay television and radio services], [b], [take rate: 4.22])
- ([jewelry, watch, clock, and silverware shops], [b], [take rate: 4.40])

In [6]:
# Match the final letter/rate fields so commas inside the category are preserved using regex
pattern = r"^[\[(]{2}(.*?)[\])]\s*,\s*[\[(]([A-Za-z])[\])]\s*,\s*[\[(]take rate:\s*([0-9.]+)[\])]{2}$" 

# Separate contents in tag 
tag_preview = merchants.tags.str.extract(pattern, flags=re.IGNORECASE)
tag_preview.columns = ["category", "letter_code", "take_rate"]
# Normalise case/whitespace and spacing around punctuation, preserving meaningful words and separators.
category_before = tag_preview.category.str.strip().str.lower()
tag_preview["category"] = (category_before.str.replace(r"\s+", " ", regex=True)
    .str.replace(r"\s*([,:])\s*", r"\1 ", regex=True).str.replace(r"\s*-\s*", " - ", regex=True).str.strip())
print(f"Category labels before/after spacing cleanup: {category_before.nunique()} → {tag_preview.category.nunique()}")
tag_preview["letter_code"] = tag_preview.letter_code.str.lower()
tag_preview["take_rate"] = pd.to_numeric(tag_preview.take_rate, errors="coerce")
display(tag_preview.head(3))

# Check parses and distributions
print(f"Incomplete parses: {tag_preview.isna().any(axis=1).sum():,} / {len(merchants):,}")
print("Letter codes:", sorted(tag_preview.letter_code.dropna().unique()))
display(tag_preview.letter_code.value_counts().head(10).rename("count").to_frame())
display(tag_preview[["take_rate"]].describe().T)
print(f"Normalised categories: {tag_preview.category.nunique()}; top 10 shown below.")
display(tag_preview.category.value_counts().head(10).rename("merchants").to_frame())

Category labels before/after spacing cleanup: 126 → 25


,category,letter_code,take_rate
0,"furniture, home furnishings and equipment shops, and manufacturers, except appliances",e,0.18
1,"cable, satellite, and other pay television and radio services",b,4.22
2,"jewelry, watch, clock, and silverware shops",b,4.40


Incomplete parses: 0 / 4,026
Letter codes: ['a', 'b', 'c', 'd', 'e']


,count
letter_code,
a,1602
b,1351
c,922
d,98
e,53


,count,mean,std,min,25%,50%,75%,max
take_rate,"4,026.00",4.40,1.78,0.10,2.97,4.50,6.03,7.00


Normalised categories: 25; top 10 shown below.


,merchants
category,
"digital goods: books, movies, music",195
artist supply and craft shops,193
"computer programming, data processing, and integrated systems design services",191
shoe shops,185
"gift, card, novelty, and souvenir shops",182
"furniture, home furnishings and equipment shops, and manufacturers, except appliances",182
"computers, computer peripheral equipment, and software",181
"florists supplies, nursery stock, and flowers",180
tent and awning shops,178


#### What might letters a–e mean?

The full category-by-letter table below shows merchant counts: read down a letter column to see its business categories. Compare the take-rate ranges as a second clue; these are observed source values, not confirmed contractual thresholds.

In [7]:
# Count merchants in each category/letter combination; a nonzero count means the category occurs in that tier.
category_letters = pd.crosstab(tag_preview.category, tag_preview.letter_code).reindex(columns=list("abcde"), fill_value=0)
print(category_letters.to_string())  # Show every category without pandas truncation.

# Compare take-rate ranges to see whether letters describe pricing rather than industry.
letter_summary = tag_preview.groupby("letter_code").agg(
    merchants=("category", "size"), categories=("category", "nunique"),
    min_take_rate=("take_rate", "min"), max_take_rate=("take_rate", "max"), mean_take_rate=("take_rate", "mean"))
display(letter_summary.round(2))
letters_per_category = category_letters.gt(0).sum(axis=1)
print(f"Categories in multiple letters: {letters_per_category.gt(1).sum()} / {len(category_letters)}; "
      f"in all five letters: {letters_per_category.eq(5).sum()}.")


letter_code                                                                             a   b   c   d  e
category                                                                                                
antique shops - sales, repairs, and restoration services                               37  54  30   7  1
art dealers and galleries                                                              20  56  34   1  1
artist supply and craft shops                                                          84  48  54   5  2
bicycle shops - sales and service                                                      68  57  39   4  2
books, periodicals, and newspapers                                                     63  61  34   3  3
cable, satellite, and other pay television and radio services                          84  49  34   5  3
computer programming, data processing, and integrated systems design services          78  54  47   8  4
computers, computer peripheral equipment, and software 

,merchants,categories,min_take_rate,max_take_rate,mean_take_rate
letter_code,,,,,
a,1602,25,5.50,7.00,6.23
b,1351,25,3.10,5.10,4.09
c,922,25,1.40,3.10,2.25
d,98,24,0.52,1.40,0.99
e,53,22,0.10,0.48,0.31


Categories in multiple letters: 25 / 25; in all five letters: 21.


**Interpretation:** the letters appear to represent **take-rate tiers**, ordered from `a` (highest) to `e` (lowest): `a` 5.50–7.00, `b` 3.10–5.10, `c` 1.40–3.10, `d` 0.52–1.40 and `e` 0.10–0.48, in the supplied take-rate units.

They are not distinct industry groups: 25 of 25 normalised categories appear under multiple letters, including 21 under all five. The earlier 126 labels included formatting duplicates; whitespace/punctuation normalisation reduces them to 25 categories. The business mix varies, but category alone does not determine the letter. The displayed boundaries overlap at 3.10 and 1.40, so do not infer exact assignment rules from the rounded rates.

This is a data-based hypothesis, not a supplied definition. The letters could label a commercial or risk tier associated with pricing; these checks cannot establish the business meaning or claim that `a` means safest/riskiest. Retain the neutral name `letter_code` until a data dictionary or tutor confirms it.

## 3. Consumers and the users

**Rows:** one consumer in `consumers`; one identifier mapping in `users`. Transactions refer to `user_id`, so reach demographics through `users.consumer_id → consumers.consumer_id`.

`postcode` is a code, not a numeric measure. Some source values from NT or ACT have only three digits due to 0-pad; later ETL should validate and pad appropriate values to four digits. Consumer postcode describes the **customer's area**, not the merchant's location.

In [8]:
# Show consumers data
display(consumers.head(3))

# Profile consumers and users data by columns
profile(consumers)
profile(users)

# Check postcode and gender values
print("Postcode string lengths:")
display(consumers.postcode.str.len().value_counts().sort_index().rename("consumers").to_frame())
display(consumers.gender.value_counts().rename("consumers").to_frame())

,name,address,state,postcode,gender,consumer_id
0,Yolanda Williams,413 Haney Gardens Apt. 742,WA,6935,Female,1195503
1,Mary Smith,3764 Amber Oval,NSW,2782,Female,179208
2,Jill Jones MD,40693 Henry Greens,NT,862,Female,1194530


,dtype,count,null,blank,distinct,range / values
column,,,,,,
name,object,499999,0,0,221377,Examples: Yolanda Williams | Mary Smith
address,object,499999,0,0,499955,Examples: 413 Haney Gardens Apt. 742 | 3764 Amber Oval
state,object,499999,0,0,8,"ACT, NSW, NT, QLD, SA, TAS, VIC, WA"
postcode,string,499999,0,0,3167,Examples: 6935 | 2782
gender,object,499999,0,0,3,"Female, Male, Undisclosed"
consumer_id,int64,499999,0,0,499999,4 → 1499995


,dtype,count,null,blank,distinct,range / values
column,,,,,,
user_id,int64,499999,0,0,499999,1 → 499999
consumer_id,int64,499999,0,0,499999,4 → 1499995


Postcode string lengths:


,consumers
postcode,
3,7909
4,492090


,consumers
gender,
Male,224979
Female,224946
Undisclosed,50074


## 4. Fraud score records

**Row:** a score for customer / merchant for an order-datetime. The intended join keys appear to be `(user_id, order_datetime)` and `(merchant_abn, order_datetime)`. Verify uniqueness before joining: repeated right-side keys can multiply transaction rows.

Observed scores are not on 0-1 scale, it probably follows a 0-100 scale instead but needs confirmation. A missing match is an **unknown score**, not zero risk. 

We will accomodate the Week 8 fraud delta as "ground truths" later.

In [9]:
# Display the head, data profile, fraud score distribution for consumers and merchants
for name, df, key in [("Consumer fraud", consumer_fraud, "user_id"),
                      ("Merchant fraud", merchant_fraud, "merchant_abn")]:
    display(Markdown(f"**{name}**"))
    display(df.head(2)) # display first two rows
    profile(df)
    display(df[["fraud_probability"]].describe().T) # distribution of fraud probability

# Check if any datasets are missing the required merchant or consumer keys for join
key_checks = []
for name, df, keys in [
    ("merchants", merchants, ["merchant_abn"]), 
    ("consumers", consumers, ["consumer_id"]),
    ("users: user_id", users, ["user_id"]), 
    ("users: consumer_id", users, ["consumer_id"]),
    ("consumer_fraud", consumer_fraud, ["user_id", "order_datetime"]),
    ("merchant_fraud", merchant_fraud, ["merchant_abn", "order_datetime"])
    ]:
    key_checks.append({"table / key": name, "null key rows": df[keys].isna().any(axis=1).sum(),
                       "extra rows with repeated key": df.duplicated(keys).sum(),
                       "exact duplicate rows": df.duplicated().sum()})
display(pd.DataFrame(key_checks).set_index("table / key"))

**Consumer fraud**

,user_id,order_datetime,fraud_probability
0,6228,2021-12-19,97.63
1,21419,2021-12-10,99.25


,dtype,count,null,blank,distinct,range / values
column,,,,,,
user_id,int64,34864,0,0,20128,1 → 24081
order_datetime,object,34864,0,0,365,2021-02-28 → 2022-02-27; 0 invalid/null dates
fraud_probability,float64,34864,0,0,34765,8.29 → 99.25


,count,mean,std,min,25%,50%,75%,max
fraud_probability,"34,864.00",15.12,9.95,8.29,9.63,11.74,16.22,99.25


**Merchant fraud**

,merchant_abn,order_datetime,fraud_probability
0,19492220327,2021-11-28,44.40
1,31334588839,2021-10-02,42.76


,dtype,count,null,blank,distinct,range / values
column,,,,,,
merchant_abn,int64,114,0,0,61,11149063370 → 99989036621
order_datetime,object,114,0,0,64,2021-03-25 → 2022-02-27; 0 invalid/null dates
fraud_probability,float64,114,0,0,113,18.21 → 94.13


,count,mean,std,min,25%,50%,75%,max
fraud_probability,114.00,40.42,17.19,18.21,28.99,32.69,48.40,94.13


,null key rows,extra rows with repeated key,exact duplicate rows
table / key,,,
merchants,0,0,0
consumers,0,0,0
users: user_id,0,0,0
users: consumer_id,0,0,0
consumer_fraud,0,99,99
merchant_fraud,0,0,0


## 5. Transactions: schema, dates and amounts

**Row:** one supplied order, with `order_id` as the candidate identifier. Each Parquet file contains four columns; `order_datetime` is a fifth column encoded in the parent folder. Reading an isolated file with pandas omits that date, so it must be restored or read using a partition-aware loader.

| Column | Meaning | Physical type |
|---|---|---|
| `user_id` | Customer key into the user bridge | int64 |
| `merchant_abn` | Merchant key | int64 |
| `dollar_value` | Order amount in dollars; currency not explicitly documented | double |
| `order_id` | Order identifier | string |
| `order_datetime` | Order date; no time-of-day supplied | folder string `YYYY-MM-DD` |

The following scan checks **every file and row**, keeping only small per-file summaries and sets of user/merchant IDs. It reports exact counts, min/max and means; no deduplication or filtering is applied here.

In [10]:
# Set expected schema
expected_schema = pq.read_schema(files.iloc[0].path).remove_metadata()
print(expected_schema)

# Check first few transaction records and add order_datetime from filename as column
first = files.iloc[0]
transaction_preview = pd.read_parquet(first.path).head(3).assign(order_datetime=first.date) 
display(transaction_preview)

user_id: int64
merchant_abn: int64
dollar_value: double
order_id: string


,user_id,merchant_abn,dollar_value,order_id,order_datetime
0,1,28000487688,133.23,0c37b3f7-c7f1-48cb-bcc7-0a58e76608ea,2021-02-28
1,18485,62191208634,79.13,9e18b913-0465-4fd4-92fd-66d15e65d93c,2021-02-28
2,1,83690644458,30.44,40a2ff69-ea34-4657-8429-df7ca957d6a1,2021-02-28


In [11]:
# Set known users and merchants from corresponding data to check if each transaction can be joined
known_users, known_merchants = set(users.user_id), set(merchants.merchant_abn)
scan_rows, user_sets, merchant_sets = [], {}, {}

# Examine through each transaction order-datetime 
for record in files.itertuples(index=False):
    # Check if schema is as expected
    schema = pq.read_schema(record.path).remove_metadata()
    if not schema.equals(expected_schema):
        raise ValueError(f"Transaction schema changed: {record.path}")

    # Read as df the transaction records of current order-datetime
    df = pd.read_parquet(record.path)
    amount = df.dollar_value
    user_sets.setdefault(record.snapshot, set()).update(df.user_id.dropna().unique())
    merchant_sets.setdefault(record.snapshot, set()).update(df.merchant_abn.dropna().unique())

    # Append transaction summary statistics of the current order-datetime
    scan_rows.append({
        "snapshot": record.snapshot, "date": pd.Timestamp(record.date), "rows": len(df), # snapshot level summaries
        "amount_count": amount.count(), "amount_sum": amount.sum(), # transaction value totals, exclude nulls 
        "amount_min": amount.min(), "amount_max": amount.max(), # transaction value min and max from this order-datetime
        "nonpositive": amount.le(0).sum(), "nonfinite": (~np.isfinite(amount) & amount.notna()).sum(), # count anomaly flags
        "unknown_user_rows": (~df.user_id.isin(known_users)).sum(), # no. transaction rows with users unseen in user data
        "unknown_merchant_rows": (~df.merchant_abn.isin(known_merchants)).sum(), # no. transaction rows with merchants unseen in merchant data
        **{f"null_{c}": df[c].isna().sum() for c in df.columns}, # null count for each column
    })

# Convert to dataframe and check no. rows
scan = pd.DataFrame(scan_rows)
assert scan.rows.sum() == files.rows.sum(), "Read counts differ from file metadata."
del df # free last loop assignment

# Display cleanly
display(scan.head(10).style.set_properties(**{"white-space": "nowrap"}))

,snapshot,date,rows,amount_count,amount_sum,amount_min,amount_max,nonpositive,nonfinite,unknown_user_rows,unknown_merchant_rows,null_user_id,null_merchant_abn,null_dollar_value,null_order_id
0,transactions_20210228_20210827_snapshot,2021-02-28 00:00:00,17107,17107,2784089.493477,0.008727,22569.520606,0,0,0,704,0,0,0,0
1,transactions_20210228_20210827_snapshot,2021-03-01 00:00:00,15170,15170,2435135.556607,0.033175,15620.935235,0,0,0,615,0,0,0,0
2,transactions_20210228_20210827_snapshot,2021-03-02 00:00:00,15417,15417,2543122.286327,0.009907,23098.558242,0,0,0,618,0,0,0,0
3,transactions_20210228_20210827_snapshot,2021-03-03 00:00:00,16093,16093,2688663.363734,0.002351,22411.822428,0,0,0,643,0,0,0,0
4,transactions_20210228_20210827_snapshot,2021-03-04 00:00:00,16988,16988,2915739.941191,0.004315,30653.307928,0,0,0,724,0,0,0,0
5,transactions_20210228_20210827_snapshot,2021-03-05 00:00:00,17697,17697,3018162.388184,0.002570,35514.546101,0,0,0,729,0,0,0,0
6,transactions_20210228_20210827_snapshot,2021-03-06 00:00:00,16924,16924,2954777.959991,0.000547,40633.527214,0,0,0,684,0,0,0,0
7,transactions_20210228_20210827_snapshot,2021-03-07 00:00:00,16798,16798,2835638.439838,0.009731,21432.879944,0,0,0,700,0,0,0,0
8,transactions_20210228_20210827_snapshot,2021-03-08 00:00:00,16262,16262,2632684.411956,0.005469,17276.753820,0,0,0,657,0,0,0,0
9,transactions_20210228_20210827_snapshot,2021-03-09 00:00:00,15972,15972,2615274.273786,0.002214,27116.056648,0,0,0,618,0,0,0,0


In [12]:
# Summarise transactions data coverage (timeframe, no. days, missing days, unique users and merchants)
coverage = scan.groupby("snapshot").agg(rows=("rows", "sum"), start=("date", "min"), end=("date", "max"), observed_days=("date", "nunique"))
coverage["missing_days_within_span"] = (coverage.end - coverage.start).dt.days + 1 - coverage.observed_days
coverage["unique_users"] = pd.Series({k: len(v) for k, v in user_sets.items()})
coverage["unique_merchants"] = pd.Series({k: len(v) for k, v in merchant_sets.items()})

# Display with snapshot shorthand labels
display(coverage.rename(index=snapshot_labels))

,rows,start,end,observed_days,missing_days_within_span,unique_users,unique_merchants
snapshot,,,,,,,
part2,3643266,2021-02-28,2021-08-27,181,0,24081,4359
part3,4508106,2021-08-28,2022-02-27,184,0,24081,4379
part4,6044133,2022-02-28,2022-10-26,241,0,24081,4395


In [13]:
# Summarise transaction (money) amounts (min, max, mean, anomaly flags)
amounts = scan.groupby("snapshot").agg(minimum=("amount_min", "min"), maximum=("amount_max", "max"),
                                        total=("amount_sum", "sum"), count=("amount_count", "sum"),
                                        nonpositive=("nonpositive", "sum"), nonfinite=("nonfinite", "sum"))
amounts["mean"] = amounts.total / amounts["count"]
display(amounts[["minimum", "mean", "maximum", "nonpositive", "nonfinite"]]
        .rename(index=snapshot_labels).assign(minimum=lambda d: d.minimum.map("{:.6g}".format)))

# Summarise transaction general anomaly flags (null in any column)
display(scan.groupby("snapshot")[[c for c in scan if c.startswith("null_")]].sum().rename(index=snapshot_labels))

# Summarise transaction involved user and merchants
all_user_ids = set.union(*user_sets.values())
all_merchant_ids = set.union(*merchant_sets.values())
display(pd.DataFrame([
    {"field": key, "minimum": min(values), "maximum": max(values), "distinct": len(values)}
    for key, values in [("user_id", all_user_ids), ("merchant_abn", all_merchant_ids)]
]).set_index("field"))

,minimum,mean,maximum,nonpositive,nonfinite
snapshot,,,,,
part2,9.75666e-08,166.28,"105,193.89",0,0
part3,6.58012e-06,166.40,"77,320.87",0,0
part4,7.3799e-07,166.07,"71,382.18",0,0


,null_user_id,null_merchant_abn,null_dollar_value,null_order_id
snapshot,,,,
part2,0,0,0,0
part3,0,0,0,0
part4,0,0,0,0


,minimum,maximum,distinct
field,,,
user_id,1,24081,24081
merchant_abn,10023283211,99990536339,4422


### Coverage checks before combining snapshots

Compare actual partition dates with the advertised filename dates. An overlap would need investigation before concatenating releases. Repeated order IDs across different dates are a separate check, still outstanding.

In [14]:
checks = []
for name, group in scan.groupby("snapshot"):
    start_text, end_text = re.search(r"transactions_(\d{8})_(\d{8})", name).groups()
    start, end = pd.to_datetime(start_text), pd.to_datetime(end_text)
    outside = (group.date < start) | (group.date > end)
    checks.append({"snapshot": name, "advertised_end": end.date(),
                   "actual_end": group.date.max().date(),
                   "rows outside advertised dates": group.loc[outside, "rows"].sum()})
display(pd.DataFrame(checks).set_index("snapshot").rename(index=snapshot_labels))
print("Dates appearing in multiple snapshots:", int((scan.groupby("date").snapshot.nunique() > 1).sum()))
all_dates = pd.date_range(scan.date.min(), scan.date.max())
print("Missing dates across all snapshots:", len(all_dates.difference(scan.date.unique())))

,advertised_end,actual_end,rows outside advertised dates
snapshot,,,
part2,2021-08-27,2021-08-27,0
part3,2022-02-27,2022-02-27,0
part4,2022-08-28,2022-10-26,1634128


Dates appearing in multiple snapshots: 0
Missing dates across all snapshots: 0


## 6. Can the tables join cleanly?

We are interested in whether the tables can join cleanly for downstream analysis and modelling. The intended path is `transactions → users → consumers`, with a separate `transactions → merchants` join. The checks below measure missing reference matches **without building a large joined table**. 

Note: a match on IDs does not guarantee at this stage that its state-postcode can be joined with ABS SA2 zone, and matched consumer / merchant IDs may not necessarily be matched to a fraud probability from supplied records.

In [15]:
# Check no. users and merchants in transaction records which are unseen in master records
display(scan.groupby("snapshot")[["unknown_user_rows", "unknown_merchant_rows"]].sum().rename(index=snapshot_labels))


,unknown_user_rows,unknown_merchant_rows
snapshot,,
part2,0,149228
part3,0,184414
part4,0,247188


In [16]:
# Check whether potential joins will work 
bridge_checks = pd.Series({
    "User mappings with no consumer record": (~users.consumer_id.isin(consumers.consumer_id)).sum(),
    "Consumers with no user mapping": (~consumers.consumer_id.isin(users.consumer_id)).sum(),
    "Consumer fraud rows with unknown user": (~consumer_fraud.user_id.isin(users.user_id)).sum(),
    "Merchant fraud rows with unknown merchant": (~merchant_fraud.merchant_abn.isin(merchants.merchant_abn)).sum(),
    "Merchants with no observed orders": len(known_merchants - set.union(*merchant_sets.values())),
    "Transaction rows with unknown merchant": int(scan.unknown_merchant_rows.sum()),
    "Unique unknown merchants in transaction rows": len(all_merchant_ids - known_merchants)
})
display(bridge_checks.rename("rows / merchants").to_frame())

,rows / merchants
User mappings with no consumer record,0
Consumers with no user mapping,0
Consumer fraud rows with unknown user,0
Merchant fraud rows with unknown merchant,19
Merchants with no observed orders,0
Transaction rows with unknown merchant,580830
Unique unknown merchants in transaction rows,396


From the `users`, `consumers`, `merchants`, `frauds`, and `transactions` data, we flag the following missing details for a join:
- 580,830 transactions (4.09%) refer to a merchant that is absent from the merchant table
- 396 (unique) merchants are referred to in transactions but absent from merchant table 
- 19 merchant fraud rows refer to a merchant that is absent from the merchant table

## 7. External-data candidate: ABS 2021 Census

A possible external dataset we could use is the Australia-wide **2021 General Community Profile, SA2**. Specifically:
- **G02 (Selected medians and averages)** for area-level income context 
- **G01 (Selected person characteristics by sex)** for population context

These tables could help describe profile the consumer base of each merchant, at an area averages (not an individual customer's income or demographic)

- [ABS DataPacks selector](https://www.abs.gov.au/census/find-census-data/datapacks): choose 2021 → General Community Profile → SA2 → Australia; [direct ZIP](https://www.abs.gov.au/census/find-census-data/datapacks/download/2021_GCP_SA2_for_AUS_short-header.zip). Use the included metadata to confirm field names, units and missing-value codes.
- [ABS table descriptions](https://www.abs.gov.au/census/find-census-data/geopackages) identify G01 and G02. The 2021 census is close to the supplied 2021–2022 transaction period; record publication availability if later simulating historical decisions.
- **Join gap:** consumers have postcode/state, but no SA2 code. Investigate a documented 2021 postcode/Postal Area-to-SA2 correspondence with allocation weights before joining. Do not assume one postcode equals one SA2 or perform an unweighted many-to-many join.
- [ABS Postal Areas guidance](https://www.abs.gov.au/statistics/standards/australian-statistical-geography-standard-asgs/edition-3-july-2021-june-2026/non-abs-structures/postal-areas) explains that POAs approximate postcodes. A POA DataPack is an alternative for postcode-level exploration, but matching coverage still needs checking.

**Next decision:** choose SA2 plus a verified correspondence, or a documented postcode approximation. Retain unmatched synthetic postcodes and report their coverage. Do not obtain merchant locations by relabelling customer areas. Links checked 6 September 2026.

### 7.1 Are zero incomes genuine or unavailable?

Inspect `Median_tot_prsnl_inc_weekly` in the downloaded ABS G02 extract before deciding which SA2 incomes can enter the weighted postcode estimate. This section reads the four small ABS extracts in `landing/abs_2021` (downloaded by ETL section 2); it does not depend on raw/curated outputs or write any data tables.

Distinguish actual NULLs, blank strings, nonnumeric codes and numeric zeros. A stored zero alone does not establish that the area's median income is genuinely $0.

In [17]:
# Profile the original income field before any ETL filtering or numeric conversion.
abs_landing = LANDING / "abs_2021"
abs_income = pd.read_parquet(abs_landing / "sa2_income")
income_text = abs_income["Median_tot_prsnl_inc_weekly"].astype("string")
income_value = pd.to_numeric(income_text, errors="coerce")
blank = income_text.str.strip().eq("").fillna(False)
income_checks = pd.Series({"SA2 rows": len(abs_income), "NULL": int(income_text.isna().sum()),
    "Blank": int(blank.sum()), "Other nonnumeric": int((income_text.notna() & ~blank & income_value.isna()).sum()),
    "Zero": int(income_value.eq(0).sum()), "Positive": int(income_value.gt(0).sum())}, name="rows")
display(income_checks.to_frame())
zero_income_sa2 = abs_income.loc[income_value.eq(0), ["SA2_CODE_2021"]]

,rows
SA2 rows,2472
NULL,0
Blank,0
Other nonnumeric,0
Zero,42
Positive,2430


### 7.2 Do zero-income SA2s affect populated postcode overlaps?

Join the mesh-block allocations and population counts directly from landing. Apply the same geographic exclusions as the ETL: omit non-spatial POAs `9494`, `9797` and `ZZZZ`, and require positive recorded population. Then identify overlaps whose SA2 has a zero income. Population counts are geographic weights, not counts of people reporting income.

In [18]:
# Reconstruct overlaps from source extracts; validate one row per mesh block in each lookup.
mb_sa2 = pd.read_parquet(abs_landing / "mb_sa2")
mb_poa = pd.read_parquet(abs_landing / "mb_poa")
mb_population = pd.read_parquet(abs_landing / "mb_population")
mesh_income_check = (mb_sa2.merge(mb_poa, on="MB_CODE_2021", how="left", validate="one_to_one")
                     .merge(mb_population, on="MB_CODE_2021", how="left", validate="one_to_one"))
mesh_income_check["population"] = pd.to_numeric(mesh_income_check["Person"], errors="coerce")
state_labels = {"1": "NSW", "2": "VIC", "3": "QLD", "4": "SA", "5": "WA", "6": "TAS", "7": "NT", "8": "ACT", "9": "OT"}
mesh_income_check["state"] = mesh_income_check.STATE_CODE_2021.map(state_labels)
geographic = (mesh_income_check.POA_CODE_2021.str.fullmatch(r"\d{4}", na=False)
              & ~mesh_income_check.POA_CODE_2021.isin(["9494", "9797"]) & mesh_income_check.population.gt(0))
zero_overlaps = (mesh_income_check.loc[geographic].merge(zero_income_sa2, on="SA2_CODE_2021", validate="many_to_one")
                 .groupby(["state", "POA_CODE_2021", "SA2_CODE_2021"], as_index=False).population.sum()
                 .rename(columns={"POA_CODE_2021": "postcode", "SA2_CODE_2021": "sa2_code"}))

# Names are labels verified against the ABS pages linked below, not join keys.
area_names = {"208031192": "Moorabbin Airport", "506011111": "Ashendon - Lesley",
              "801101145": "Molonglo - East", "506021121": "Perth Airport"}
zero_overlaps["sa2_name"] = zero_overlaps.sa2_code.map(area_names)
display(zero_overlaps[["state", "postcode", "sa2_code", "sa2_name", "population"]])
print(f"{zero_overlaps.sa2_code.nunique()} zero-income SA2s; "
      f"{len(zero_overlaps[['state', 'postcode']].drop_duplicates())} state/postcode pairs; "
      f"{zero_overlaps.population.sum():,.0f} recorded residents.")


,state,postcode,sa2_code,sa2_name,population
0,ACT,2611,801101145,Molonglo - East,4.00
1,VIC,3194,208031192,Moorabbin Airport,26.00
2,WA,6105,506021121,Perth Airport,20.00
3,WA,6111,506011111,Ashendon - Lesley,3.00


4 zero-income SA2s; 4 state/postcode pairs; 53 recorded residents.


### 7.3 Evidence and ETL decision

**Observed:** 2,472 SA2 income records contain no NULLs, blanks or nonnumeric codes, but 42 zeros. Only four zero-income SA2s contribute positive population to geographic postcode overlaps, representing 53 recorded residents across four state/postcode pairs.

ABS provides no QuickStats for each of these four areas because there are no people or a very low population:

- [Moorabbin Airport (208031192)](https://www.abs.gov.au/census/find-census-data/quickstats/2021/208031192)
- [Ashendon - Lesley (506011111)](https://www.abs.gov.au/census/find-census-data/quickstats/2021/506011111)
- [Molonglo - East (801101145)](https://www.abs.gov.au/census/find-census-data/quickstats/2021/801101145)
- [Perth Airport (506021121)](https://www.abs.gov.au/census/find-census-data/quickstats/2021/506021121)

Another example is [West Melbourne - Industrial (206041127)](https://www.abs.gov.au/census/find-census-data/quickstats/2021/206041127), which has zero income in G02 and the same ABS notice. Sources checked 7 September 2026.

**Interpretation:** these notices support treating the relevant zeros as unverified/unavailable estimates, rather than confirmed $0 medians. They do not establish a universal ABS rule that every zero encodes NULL.

**Decision:** keep source zeros in landing/raw, but retain the ETL's `income > 0` eligibility rule for the weighted income estimate. Exclude an ineligible SA2's income and its overlap population from the numerator and denominator; keep its population in the overall coverage measure. This excludes an income contribution, not consumers or transactions. If no usable population remains, leave district income NULL. A future confirmed genuine zero should contribute zero to the numerator and its population to the denominator.

The estimate remains a population-weighted **average of SA2 medians**, not a postcode median or an individual's income.

## 8. What this suggests for ETL

These are **proposals for review**, not transformations performed by this notebook. The layer names are our project convention; the spec requires an automated ingestion pipeline and curated outputs without prescribing these exact names.

| Layer | Suggested row unit and responsibility |
|---|---|
| **Landing** | Preserve the original data and ABS downloads unchanged. |
| **Raw** | Separate source tables: merchants, consumers, user, fraud, and transactions. Apply correct delimiters, explicitly add ABN from index to column, recover partition dates, define types and retain source-file. Flag filename/date discrepancies. |
| **Curated** | Validated consumer/merchant tables and left-joined transactions with consumer/merchant (entity) information. Parse merchant tags, standardise validated postcodes, resolve repeated fraud keys before joins, add risk/quality flags, verify row-count preservation and external-data coverage. Retain unknowns explicitly. |
| **Analysis** | One row per merchant for ranking features (optionally merchant × period for trends): order count/value, customer reach, take-rate-based revenue after unit confirmation, risk and customer-area context. Left-join on merchants table so those with no orders remain eligible for consideration. |

Before editing `01_etl`:

1. Confirm score/take-rate units and the letter code meaning. The 99 extra consumer-fraud key repeats are also exact duplicate rows; confirm and document a deduplication rule.
2. Decide how to handle the final snapshot's extra dates and validate `order_id` uniqueness within/across releases. Extreme amounts should be investigated, not automatically dropped.
3. Investigate the unmatched merchant ABNs; retain and flag these orders. Choose the external geography route and measure unmatched rows after joining. Fraud coverage by entity **and date** also remains to be measured; supplied scores end before the final transactions.